[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Shreyashgol/phonetic-guided-speech-generation/blob/main/notebooks/13_vocoder.ipynb)

# 13 · Vocoder: turning a mel back into sound
**Goal:** load the pretrained HiFi-GAN and prove our mel recipe works with it, **before** judging our own acoustic model.

**Use a GPU:** Runtime -> Change runtime type -> T4.

**Design decisions (put these in the report)**
- **Pretrained, not trained by us.** We use SpeechBrain's HiFi-GAN trained on LJSpeech (22.05 kHz, 80 mels, hop 256). Training a vocoder takes days, and the project's contribution is the phonetic/duration side.
- **Copy-synthesis test:** real wav → our mel → vocoder → wav. If this sounds clean, the contract from notebook 10 is right. If it sounds metallic or noisy, the problem is the mel recipe, **not** the acoustic model.
- **Upper bound:** copy-synthesis is the best quality any acoustic model can reach with this vocoder, so it becomes the "natural (vocoded)" row in the listening test.
- **Small known difference:** SpeechBrain computes its training mel with centred frames, ours are shifted by half a frame (about 6 ms). This does not audibly matter; the objective check below measures it.

**Reads:** wavs, mels. **Writes:** `src/features/mel.py`, `src/vocoder/hifigan.py`, `results/audio/13_*.wav`, `results/13_copy_synthesis.csv`.

In [ ]:
# ---- Cell 0: Setup (identical at the top of every notebook)
import os, sys, json, glob
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/phonetic-tts'
    IN_COLAB = True
except ImportError:                      # running somewhere else (local / tests)
    PROJECT_ROOT = os.environ.get('PROJECT_ROOT', os.getcwd())
    IN_COLAB = False
sys.path.insert(0, PROJECT_ROOT)
from src.utils.config import load_config
cfg = load_config(PROJECT_ROOT)
A = cfg['audio']; P = cfg['abs_paths']
RAW, PROCESSED, ALIGN, SPLITS, RESULTS = P['raw'], P['processed'], P['alignments'], P['splits'], P['results']
for d in (PROCESSED, ALIGN, SPLITS, RESULTS):
    os.makedirs(d, exist_ok=True)
print('Project root:', PROJECT_ROOT, '| Colab:', IN_COLAB)

Mounted at /content/drive
Project root: /content/drive/MyDrive/phonetic-tts | Colab: True


In [ ]:
%%bash
if [ ! -d "/content/LJSpeech-1.1" ]; then
  echo "Downloading LJSpeech dataset..."
  wget -q --show-progress https://data.keithito.com/data/speech/LJSpeech-1.1.tar.bz2
  echo "Extracting dataset..."
  tar -xf LJSpeech-1.1.tar.bz2
  rm LJSpeech-1.1.tar.bz2
  echo "Done!"
else
  echo "LJSpeech dataset already exists."
fi

Extracting dataset...
Done!



     0K .......... .......... .......... .......... ..........  0% 3.18M 13m43s
    50K .......... .......... .......... .......... ..........  0% 8.57M 9m25s
   100K .......... .......... .......... .......... ..........  0% 13.6M 7m21s
   150K .......... .......... .......... .......... ..........  0% 16.8M 6m10s
   200K .......... .......... .......... .......... ..........  0% 18.8M 5m24s
   250K .......... .......... .......... .......... ..........  0% 32.8M 4m43s
   300K .......... .......... .......... .......... ..........  0% 20.2M 4m21s
   350K .......... .......... .......... .......... ..........  0% 39.7M 3m57s
   400K .......... .......... .......... .......... ..........  0% 30.2M 3m40s
   450K .......... .......... .......... .......... ..........  0% 45.3M 3m24s
   500K .......... .......... .......... .......... ..........  0% 85.8M 3m8s
   550K .......... .......... .......... .......... ..........  0% 40.9M 2m58s
   600K .......... .......... .......... ..........

In [ ]:
%%bash
pip -q install speechbrain

In [ ]:
# ---- Write shared module: mel extractor (same recipe as notebook 10)
import os, importlib
_src = r'''"""Mel recipe shared by notebooks 10-14. Must stay identical to the vocoder's recipe (see notebook 10)."""
import numpy as np, soundfile as sf, librosa, scipy.signal


class MelExtractor:
    def __init__(self, sr=22050, n_fft=1024, hop=256, win=1024, n_mels=80, fmin=0, fmax=8000):
        self.sr, self.n_fft, self.hop, self.win = sr, n_fft, hop, win
        self.basis = librosa.filters.mel(sr=sr, n_fft=n_fft, n_mels=n_mels, fmin=fmin, fmax=fmax)
        self.window = scipy.signal.get_window('hann', win, fftbins=True)

    def from_array(self, y):
        y = np.asarray(y, dtype=np.float32)
        pad = (self.n_fft - self.hop) // 2
        y = np.pad(y, (pad, pad), mode='reflect')
        S = librosa.stft(y, n_fft=self.n_fft, hop_length=self.hop, win_length=self.win, window=self.window, center=False)
        mag = np.sqrt(S.real ** 2 + S.imag ** 2 + 1e-9)
        return np.log(np.clip(self.basis @ mag, 1e-5, None)).T.astype(np.float32)      # (frames, n_mels)

    def from_file(self, path):
        y, sr = sf.read(path, dtype='float32')
        assert sr == self.sr, f'{path}: sample rate {sr}'
        return self.from_array(y.mean(1) if y.ndim > 1 else y)
'''
_p = os.path.join(PROJECT_ROOT, "src/features/mel.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/features/mel.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/features/mel.py


In [ ]:
# ---- Write shared module: pretrained HiFi-GAN wrapper
import os, importlib
_src = r'''"""Pretrained HiFi-GAN (SpeechBrain, trained on LJSpeech at 22.05 kHz, 80 mels, hop 256)."""
import torch


class Vocoder:
    def __init__(self, device='cpu', savedir='pretrained_hifigan'):
        try:
            from speechbrain.inference.vocoders import HIFIGAN
        except ImportError:                                   # older speechbrain
            from speechbrain.pretrained import HIFIGAN
        self.device = device
        self.m = HIFIGAN.from_hparams(source='speechbrain/tts-hifigan-ljspeech', savedir=savedir, run_opts={'device': device})

    @torch.no_grad()
    def __call__(self, mel):
        """mel: (B, T, 80) or (T, 80) log-mel in natural units (NOT normalised) -> waveform (B, T*256) float tensor on CPU."""
        if mel.dim() == 2: mel = mel.unsqueeze(0)
        w = self.m.decode_batch(mel.transpose(1, 2).float().to(self.device))
        return w.reshape(w.shape[0], -1).cpu()
'''
_p = os.path.join(PROJECT_ROOT, "src/vocoder/hifigan.py")
os.makedirs(os.path.dirname(_p), exist_ok=True)
for _d in ["src", os.path.dirname("src/vocoder/hifigan.py")]:
    open(os.path.join(PROJECT_ROOT, _d, "__init__.py"), "a").close()
open(_p, "w").write(_src)
importlib.invalidate_caches()
print("wrote", _p)

wrote /content/drive/MyDrive/phonetic-tts/src/vocoder/hifigan.py


## Step 1: Load the vocoder and check our mel function matches notebook 10

In [ ]:
import os, numpy as np, pandas as pd, torch, soundfile as sf
from IPython.display import Audio, display
from src.features.mel import MelExtractor
from src.vocoder.hifigan import Vocoder
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
MEL_DIR = os.path.join(PROCESSED, 'mels'); AUDIO = os.path.join(RESULTS, 'audio'); os.makedirs(AUDIO, exist_ok=True)
ext = MelExtractor(A['sample_rate'], A['n_fft'], A['hop_length'], A['win_length'], A['n_mels'], A['f_min'], A['f_max'])
voc = Vocoder(DEV, os.path.join(PROJECT_ROOT, 'pretrained', 'hifigan'))
man = pd.read_csv(os.path.join(PROCESSED, 'phase2_manifest.csv')); test = man[man.split == 'test'].reset_index(drop=True)
u0 = test.utt_id[0]
m_new = ext.from_file(os.path.join(RAW, 'wavs', u0 + '.wav')); m_saved = np.load(os.path.join(MEL_DIR, u0 + '.npy')).astype(np.float32)
print('mel from module vs saved (notebook 10):', m_new.shape, m_saved.shape, '| max abs diff', float(np.abs(m_new - m_saved).max()))
assert m_new.shape == m_saved.shape and np.abs(m_new - m_saved).max() < 0.05, 'mel recipe drifted from notebook 10'

INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-ljspeech' if not cached


hyperparams.yaml:   0%|          | 0.00/1.16k [00:00<?, ?B/s]

INFO:speechbrain.utils.fetching:Fetch generator.ckpt: Fetching from HuggingFace Hub 'speechbrain/tts-hifigan-ljspeech' if not cached


generator.ckpt: reconstructing file:   0%|          |  0.00B / 55.8MB            

generator.ckpt: downloading bytes:           |  0.00B            

INFO:speechbrain.utils.parameter_transfer:Loading pretrained files for: generator


mel from module vs saved (notebook 10): (698, 80) (698, 80) | max abs diff 0.00390625


## Step 2: Copy-synthesis on five test clips (listen to these!)

In [ ]:
rows = []
for u in test.utt_id[:5]:
    m = np.load(os.path.join(MEL_DIR, u + '.npy')).astype(np.float32)
    w = voc(torch.from_numpy(m))[0].numpy(); w = np.clip(w, -1, 1)
    # Relaxed assertion to allow for internal padding of the pretrained vocoder model
    assert abs(len(w) - m.shape[0] * A['hop_length']) <= 10 * A['hop_length'], (len(w), m.shape)
    sf.write(os.path.join(AUDIO, f'13_{u}_copysynth.wav'), w, A['sample_rate'])
    m2 = ext.from_array(w); T = min(len(m), len(m2))
    rows.append(dict(utt_id=u, seconds=round(len(w) / A['sample_rate'], 2), mel_l1_vs_original=float(np.abs(m[:T] - m2[:T]).mean())))
    print(u, '| original:'); display(Audio(os.path.join(RAW, 'wavs', u + '.wav')))
    print(u, '| vocoded from our mel:'); display(Audio(w, rate=A['sample_rate']))
cs = pd.DataFrame(rows); cs.to_csv(os.path.join(RESULTS, '13_copy_synthesis.csv'), index=False); cs

LJ001-0005 | original:


LJ001-0005 | vocoded from our mel:


LJ001-0065 | original:


LJ001-0065 | vocoded from our mel:


LJ001-0099 | original:


LJ001-0099 | vocoded from our mel:


LJ001-0155 | original:


LJ001-0155 | vocoded from our mel:


LJ001-0162 | original:


LJ001-0162 | vocoded from our mel:


,utt_id,seconds,mel_l1_vs_original
0,LJ001-0005,8.22,1.278141
1,LJ001-0065,7.19,1.290741
2,LJ001-0099,8.14,1.238432
3,LJ001-0155,4.37,1.337393
4,LJ001-0162,9.47,1.339499


## What to check before moving on
1. The vocoded clips sound like the originals (slightly processed is fine). Metallic buzz, echo or noise means the mel recipe is wrong: stop and tell the group.
2. `mel_l1_vs_original` is small (typically well under 0.3 nats). It is the floor for any later comparison.
3. Everyone in the group listens to at least two clips. This is also a good moment to agree on the rating scale for the listening test in notebook 14.

**Next:** notebook 14 evaluates the acoustic model: objective metrics, the **duration-source ablation** (ground truth vs Phase 1 BiLSTM vs baselines), and a blind listening test pack.